Run Simulation
-for each wind input, run the power model 10 times and for each run of the power model, run SWMM once
-put power model run and swmm run in same loop
-for loop for wind inputs (we can put them in wind_inputs folder)
-for each wind input, run power model 10x
    -for each power model run swmm 1x
    
to do: clean up power model results and put them in a folder together so that for swmm, we can loop through that folder

In [ ]:
import pandas as pd
import numpy as np
import subprocess
import os
from pathlib import Path

#seed
np.random.seed(42)

# Set base paths
#script_dir = os.getcwd()
script_dir = r"C:\Users\csofios\Desktop\glwa-main" #PATH ADJUSTED
script_path = os.path.join(script_dir, 'glwa-main', 'network_simulation_wind.py')
roads_path = os.path.join(script_dir, 'glwa-main', 'runs', 'glwa_area', 'inputs', 'wayne_plus_surrounding_roads_2.24.24.shp')
buildings_path = os.path.join(script_dir, 'glwa-main', 'runs', 'glwa_area', 'inputs', 'buildings_subs_pumps_3.28.24.csv')
outage_file = os.path.join(script_dir, 'pump_outages_single_iteration.csv')
wind_input_folder = os.path.join(script_dir, 'glwa-main', 'wind_inputs')

# Layout input lives directly in script_dir
layout_folder = script_dir

# Where to save simulation results (your desired folder)
#CHANGE THE OUTPUT_FOLDER EACH TIME YOU RUN A NEW SIMULATION
output_folder = os.path.join(script_dir, 'sim_outputs_1020wind')
Path(output_folder).mkdir(parents=True, exist_ok=True)

# Where to save final pump status CSVs
output_root = os.path.join(script_dir, 'outputs')
Path(output_root).mkdir(parents=True, exist_ok=True)

# Loop through all wind CSVs
for wind_csv in Path(wind_input_folder).glob("*.csv"):
    for i in range(5):  # Run each wind CSV 100 times
        print(f"\n▶️ Running {wind_csv.name}, iteration {i+1}")

        # Full result path for this run
        result_file = f'Layout_sub_405output_run{i+1}'
        output_result_path = os.path.join(output_folder, result_file)

        # Run simulation subprocess
        try:
            result = subprocess.run(
                [
                    r'C:\Users\csofios\.conda\envs\swmm_env\python.exe',
                    script_path,
                    roads_path,
                    buildings_path,
                    layout_folder,          # ✅ layout input folder (has ntw_cust_sub.npz)
                    output_result_path,     # ✅ full output result path (not just a filename!)
                    '1',
                    '--wind_csv', str(wind_csv),
                ],
                check=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                universal_newlines=True
            )
        except subprocess.CalledProcessError as e:
            print("❌ Simulation script failed!")
            print("Command:", e.cmd)
            print("Exit Code:", e.returncode)
            print("Standard Output:\n", e.stdout)
            print("Standard Error:\n", e.stderr)
            continue  # Skip to next run

        # Process simulation output
        try:
            with open(output_result_path, 'r') as f:
                lines = [line.strip() for line in f.readlines()]
            pump_lines = [line for line in lines if ',Pump,' in line]
            pump_data = [line.split(',') for line in pump_lines]

            pump_df = pd.DataFrame(pump_data, columns=['ID', 'Type', 'Latitude', 'Longitude', 'OutageProbability'])
            pump_df['ID'] = pump_df['ID'].astype(float).astype(int).astype(str)
            pump_df['OutageProbability'] = pump_df['OutageProbability'].astype(float)

            def convert_probability(prob):
                return 1 if np.random.uniform(0, 1) > prob else 0

            pump_df['Status'] = pump_df['OutageProbability'].apply(convert_probability)

            pump_outages_df = pd.read_csv(outage_file)
            pump_outages_df['ID'] = pump_outages_df['ID'].astype(str)

            final_df = pump_outages_df[['Name', 'ID']].merge(pump_df[['ID', 'Status']], on='ID', how='inner')

            # Save final merged result
            save_dir = os.path.join(output_root, 'pump status')
            Path(save_dir).mkdir(parents=True, exist_ok=True)

            output_csv_path = os.path.join(save_dir, f'pump_status_{i+1}.csv')
            final_df.to_csv(output_csv_path, index=False)

            print(f"✅ Saved: {output_csv_path}")

        except Exception as ex:
            print(f"⚠️ Failed to process pump status for {wind_csv.name} run {i+1}: {ex}")



▶️ Running wind_inputs.csv, iteration 1
❌ Simulation script failed!
Command: ['C:\\Users\\csofios\\.conda\\envs\\swmm_env\\python.exe', 'C:\\Users\\csofios\\Desktop\\glwa-main\\glwa-main\\network_simulation_wind.py', 'C:\\Users\\csofios\\Desktop\\glwa-main\\glwa-main\\runs\\glwa_area\\inputs\\wayne_plus_surrounding_roads_2.24.24.shp', 'C:\\Users\\csofios\\Desktop\\glwa-main\\glwa-main\\runs\\glwa_area\\inputs\\buildings_subs_pumps_3.28.24.csv', 'C:\\Users\\csofios\\Desktop\\glwa-main', 'C:\\Users\\csofios\\Desktop\\glwa-main\\sim_outputs_1020wind\\Layout_sub_405output_run1', '1', '--wind_csv', 'C:\\Users\\csofios\\Desktop\\glwa-main\\glwa-main\\wind_inputs\\wind_inputs.csv']
Exit Code: 1
Standard Output:
 0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98


Extract Pole Locations

In [3]:
import sys
sys.executable


'C:\\Users\\csofios\\.conda\\envs\\swmm_env\\python.exe'

In [4]:
print(script_dir)

NameError: name 'script_dir' is not defined

In [2]:
print(output_folder)

NameError: name 'output_folder' is not defined